> **ARCHIVE — superseded.** This notebook is kept as a record of the research path, not as working code. Credentials were removed and outputs cleared. Results in it may rest on defects documented in `docs/RESEARCH_LOG.md` (stage: v0, LLM-named 82-name universe). Comments are partly in Vietnamese. The maintained implementation is `src/memescan/`.

# Meme Entry v2 — Absorption entry (volume spike + giá đứng)

**Dùng LẠI data cache của notebook chính** (`meme_cache/bt_daily.parquet`, `short_interest.parquet`).
Giữ nguyên ý tưởng watchlist + context; chỉ ĐỔI logic ENTRY cho đơn giản & path-dependent:

> **Entry**: ngày có **RVOL spike siêu mạnh NHƯNG giá đứng yên** (= hấp thụ/accumulation) trong vài ngày gần,
> rồi **`dist_60low` và `vs_ma20` cùng dip đột ngột sau spike** (pullback), và giá đang ngóc lên (curl).
> → vào trong *zone* của 3 indicator này. **Exit giữ nguyên** (hard stop + ATR trail + hold-monitor).
> Sau khi xác định ngày entry trên daily → **nhìn minute bars** để canh điểm vào (VWAP reclaim).

⚠️ Cần đã chạy notebook chính tới Cell 8.1 (tạo `bt_daily.parquet`) và 9.1 (short interest) trước.

In [ ]:
# Cell 1 — setup + LOAD lại data cache (y hệt pipeline cũ)
import os, time, json, logging
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("entry_v2")
CACHE_DIR = Path("./meme_cache")

ALPACA = {"key": os.getenv("ALPACA_API_KEY", ""),
          "secret": os.getenv("ALPACA_SECRET_KEY", "")}
ALPACA_HEADERS = {"APCA-API-KEY-ID": ALPACA["key"], "APCA-API-SECRET-KEY": ALPACA["secret"]}

def load_df(path):
    path = Path(path)
    if path.exists():
        try: return pd.read_parquet(path)
        except Exception: pass
    pk = Path(str(path) + ".pkl")
    return pd.read_pickle(pk) if pk.exists() else None

bt = load_df(CACHE_DIR / "bt_daily.parquet")
si_all = load_df(CACHE_DIR / "short_interest.parquet")
assert bt is not None and len(bt), "Chưa có bt_daily.parquet — chạy Cell 8.1 ở notebook chính trước."
print("bt:", len(bt), "rows |", bt["ticker"].nunique(), "tickers")
print("short interest:", 0 if si_all is None else si_all["ticker"].nunique(), "tickers")

In [ ]:
# Cell 2 — helper y hệt pipeline cũ (exit / portfolio / event / metric) để so sánh táo-táo
si_map = {tk: g.sort_values("date") for tk, g in si_all.groupby("ticker")} if (si_all is not None and len(si_all)) else {}
SI_PUB_LAG_D = 12

def si_asof(sym, d):
    g = si_map.get(sym)
    if g is None: return None
    dd = pd.to_datetime(d)
    if getattr(dd, "tzinfo", None) is not None: dd = dd.tz_localize(None)
    av = g[g["date"] + pd.Timedelta(days=SI_PUB_LAG_D) <= dd]
    if len(av) == 0: return None
    last = av.iloc[-1]; rising = bool(len(av) >= 2 and last["interest"] > av.iloc[-2]["interest"])
    return (last["dtc"], rising)

BT = {"atr_win": 14, "atr_mult": 3.0, "ma_exit": 10, "max_hold": 60,
      "max_positions": 10, "alloc": 10000.0, "start_capital": 100000.0}
EXIT_F = {"hard_stop": 0.08, "hold_monitor": True, "hold_require_si": True, "hold_below_ma": None,
          "use_si_gate": True, "si_dtc_min": 2.0}

def _atr(high, low, close, win):
    n = len(close); tr = np.zeros(n)
    for i in range(1, n): tr[i] = max(high[i]-low[i], abs(high[i]-close[i-1]), abs(low[i]-close[i-1]))
    a = np.full(n, np.nan)
    for i in range(win, n): a[i] = tr[i-win+1:i+1].mean()
    return a

def _si_gate_ok(sym, d, F):
    if not F.get("use_si_gate") or not si_map: return True
    si = si_asof(sym, d)
    if not si: return False
    dtc, rising = si
    ok = (dtc is not None) and not (isinstance(dtc, float) and np.isnan(dtc)) and dtc >= F["si_dtc_min"]
    return bool(ok or rising)

def _exit_after_entry(sym, g, close, atr, i, F, max_hold):
    entry = close[i]; peak = entry; xi = min(i + max_hold, len(close) - 1)
    for j in range(i + 1, xi + 1):
        if close[j] <= entry * (1 - F["hard_stop"]): return j
        if F.get("hold_monitor"):
            if F.get("hold_require_si") and not _si_gate_ok(sym, g["date"].iloc[j], F): return j
            hb = F.get("hold_below_ma")
            if hb and close[j] < close[max(0, j-hb+1):j+1].mean(): return j
        peak = max(peak, close[j])
        if close[j] <= peak - BT["atr_mult"] * (atr[j] if not np.isnan(atr[j]) else 0): return j
        if close[j] > entry and close[j] < close[max(0, j-BT["ma_exit"]+1):j+1].mean(): return j
    return xi

def run_portfolio(trades):
    rows, opens = [], []
    for _, tr in trades.iterrows():
        opens = [d for d in opens if d > tr["entry_date"]]
        if len(opens) >= BT["max_positions"]: continue
        opens.append(tr["exit_date"]); rows.append({**tr.to_dict(), "pnl": BT["alloc"] * tr["ret"]})
    tk = pd.DataFrame(rows)
    if len(tk):
        tk = tk.sort_values("exit_date").reset_index(drop=True); tk["equity"] = BT["start_capital"] + tk["pnl"].cumsum()
    return tk

def detect_meme_events(bt_df, H=20, surge_th=0.50):
    evs = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True); c = g["close"].values.astype(float); n = len(g); t = 0
        while t < n - 1:
            w = c[t+1:min(t+1+H, n)]
            if len(w) == 0: break
            if w.max()/c[t]-1 >= surge_th:
                pk = t+1+int(np.argmax(w))
                evs.append({"ticker": sym, "ignition_date": g["date"].iloc[t], "peak_date": g["date"].iloc[pk],
                            "base_px": c[t], "peak_px": c[pk], "run_pct": c[pk]/c[t]-1}); t = pk+1
            else: t += 1
    return pd.DataFrame(evs)

def detection_lag(td, events):
    if td is None or len(td) == 0 or len(events) == 0: return (0, len(events), np.nan)
    s = td.copy(); s["entry_date"] = pd.to_datetime(s["entry_date"]); lags = []
    for _, ev in events.iterrows():
        m = s[(s["ticker"] == ev["ticker"]) & (s["entry_date"] >= ev["ignition_date"]-pd.Timedelta(days=15)) & (s["entry_date"] <= ev["peak_date"])]
        if len(m): lags.append((m["entry_date"].min() - ev["ignition_date"]).days)
    return (len(lags), len(events), np.median(lags) if lags else np.nan)

def entry_premium_over_base(td, events):
    if len(td) == 0 or len(events) == 0: return np.nan
    p = []
    for _, tr in td.iterrows():
        e = events[(events["ticker"] == tr["ticker"]) & (events["ignition_date"] <= pd.to_datetime(tr["exit_date"])) & (events["peak_date"] >= pd.to_datetime(tr["entry_date"]))]
        if len(e):
            b = e.sort_values("ignition_date").iloc[0]["base_px"]
            if b > 0: p.append(tr["entry_px"]/b - 1)
    return np.median(p) if p else np.nan

print("helpers OK | si_map:", len(si_map), "tickers")

In [ ]:
# Cell 2b — IGNITION dates cho MỖI stock (để bạn check đúng đợt meme không)
events = detect_meme_events(bt)
print(f"Tổng {len(events)} đợt meme (>=50%/20 ngày) trên {events['ticker'].nunique()} stock\n")
for sym, g in events.sort_values(["ticker", "ignition_date"]).groupby("ticker"):
    spans = ", ".join(f"{pd.to_datetime(r.ignition_date).date()}(+{r.run_pct:.0%})" for _, r in g.iterrows())
    print(f"  {sym:6s} [{len(g)}]: {spans}")


## Entry mới — Absorption

**3 indicator/ngày** (đo từng ngày cho mỗi stock):
- `rvol`  = volume / TB20
- `dist_60low` = giá cao hơn đáy 60 ngày bao nhiêu
- `vs_ma20` = giá vs MA20

**Vào lệnh khi (zone của 3 cái):**
1. Gần đây có ngày **`rvol` spike siêu mạnh nhưng `|đổi giá|` nhỏ** → hấp thụ (volume vào mà giá đứng).
2. Hiện tại **`dist_60low` VÀ `vs_ma20` cùng dip** so với vài ngày trước → pullback sau spike.
3. Giá đang **ngóc lên** (curl) + `rvol` hiện tại chưa nổ + short-fuel còn (SI gate).

In [ ]:
# Cell 3 — ENTRY absorption (tham số ĐƠN GIẢN) + backtest, so với meme events
ENTRY = {
    "spike_rvol":   4.0,   # RVOL spike siêu mạnh
    "flat_max":     0.04,  # |đổi giá| ngày spike < 4%  -> volume cao mà giá ĐỨNG = absorption
    "lookback":     7,     # tìm ngày absorption trong 7 ngày gần
    "dip_ago":      3,     # so với 3 ngày trước để thấy dist & vs_ma20 cùng DIP
    "rvol_now_max": 3.0,   # vào khi rvol HÔM NAY chưa nổ (chưa tới ngày breakout)
}

def gen_absorption_trades(bt_df, P=ENTRY, F=EXIT_F):
    trades = []
    for sym, g in bt_df.groupby("ticker"):
        g = g.sort_values("date").reset_index(drop=True)
        if len(g) < 70: continue
        close = g["close"].values.astype(float); high = g["high"].values.astype(float)
        low = g["low"].values.astype(float); vol = g["vol"].values.astype(float); n = len(g)
        atr = _atr(high, low, close, BT["atr_win"])
        rvol = np.full(n, np.nan); ret1 = np.full(n, np.nan); dist = np.full(n, np.nan); vsma = np.full(n, np.nan)
        for k in range(60, n):
            a = vol[k-20:k].mean(); rvol[k] = vol[k]/a if a > 0 else np.nan
            ret1[k] = close[k]/close[k-1] - 1
            dist[k] = close[k]/low[k-60:k].min() - 1
            vsma[k] = close[k]/close[k-20:k].mean() - 1
        i = 64
        while i < n - 1:
            w0 = max(60, i - P["lookback"])
            absorb = any(rvol[k] > P["spike_rvol"] and abs(ret1[k]) < P["flat_max"] for k in range(w0, i))
            dip = (dist[i] < dist[i-P["dip_ago"]]) and (vsma[i] < vsma[i-P["dip_ago"]])
            early = rvol[i] < P["rvol_now_max"]
            curl = close[i] > close[i-1]
            if absorb and dip and early and curl and _si_gate_ok(sym, g["date"].iloc[i], F):
                j = _exit_after_entry(sym, g, close, atr, i, F, BT["max_hold"])
                trades.append({"ticker": sym, "entry_date": g["date"].iloc[i], "exit_date": g["date"].iloc[j],
                               "entry_px": close[i], "exit_px": close[j], "ret": close[j]/close[i]-1, "days": j-i,
                               "rvol_in": rvol[i], "dist_in": dist[i], "vsma_in": vsma[i]})
                i = j + 1; continue
            i += 1
    return pd.DataFrame(trades).sort_values("entry_date").reset_index(drop=True) if trades else pd.DataFrame()

ab = gen_absorption_trades(bt)
events = detect_meme_events(bt)
print("Absorption trades:", len(ab))
if len(ab):
    tk = run_portfolio(ab); nd, ne, ml = detection_lag(ab, events); prem = entry_premium_over_base(ab, events)
    final = tk["equity"].iloc[-1] if len(tk) else BT["start_capital"]
    print(f"detect {nd}/{ne} | lag {ml:+.0f}d | win {(ab['ret']>0).mean():.0%} | "
          f"avg {ab['ret'].mean():+.1%} | entry+nền {prem:+.0%} | final ${final:,.0f}")
    print(ab.tail(12).assign(entry_date=lambda x: pd.to_datetime(x.entry_date).dt.date)[
          ["ticker","entry_date","entry_px","ret","days","rvol_in","dist_in","vsma_in"]].to_string(index=False))

## Backtest đầy đủ

**KHÔNG look-ahead — kiểm bằng tay:**
- Entry tại ngày `i` chỉ dùng bar `<= i`: `rvol[k<i]` (absorption), `dist/vsma[i]` (low/MA quá khứ), `dip` so `i` vs `i-3`, `si_asof` có publication-lag 12 ngày. Vào tại `close[i]`.
- Exit quét bar tương lai = trade ĐANG diễn tiến (không dùng tương lai để *quyết định vào*).
- `detect_meme_events` nhìn forward NHƯNG chỉ để **chấm điểm timing**, KHÔNG tạo lệnh.
- Short interest đã trừ lag công bố → point-in-time.

In [ ]:
# Cell 5 — BACKTEST đầy đủ (no look-ahead: entry chỉ dùng bar <= i; events chỉ CHẤM ĐIỂM)
if len(ab) == 0:
    print("Chưa có absorption trade — nới ENTRY ở Cell 3.")
else:
    tk = run_portfolio(ab).copy()
    tk["entry_date"] = pd.to_datetime(tk["entry_date"]); tk["exit_date"] = pd.to_datetime(tk["exit_date"])
    tk["year"] = tk["entry_date"].dt.year
    r = tk["ret"]; final = tk["equity"].iloc[-1]
    wins, losses = tk[r > 0], tk[r <= 0]
    span = max((tk["entry_date"].max() - tk["entry_date"].min()).days / 365.25, 0.5)
    sharpe = (r.mean() / r.std(ddof=1)) if len(r) > 1 and r.std(ddof=1) > 0 else np.nan
    eq = tk["equity"].values; peak = np.maximum.accumulate(eq); dd = ((eq - peak) / peak).min()
    pf = (wins["pnl"].sum() / abs(losses["pnl"].sum())) if (len(losses) and losses["pnl"].sum() != 0) else float("inf")

    print("=" * 56); print("PERFORMANCE — Absorption entry"); print("=" * 56)
    print(f"  Trades {len(tk)} | Win {(r>0).mean():.0%} | avg win {wins['ret'].mean():+.1%} / loss {losses['ret'].mean():+.1%}")
    print(f"  Expectancy {r.mean():+.2%}/trade | Sharpe(trade) {sharpe:.2f} | ann~ {sharpe*np.sqrt(len(tk)/span):.2f}")
    print(f"  Profit factor {pf:.2f} | MaxDD {dd:.0%} | Final ${final:,.0f} ({final/BT['start_capital']-1:+.0%})")

    def tlabel(row):
        e = events[(events.ticker == row.ticker) & (events.ignition_date <= row.exit_date) & (events.peak_date >= row.entry_date)]
        if len(e) == 0: return "no-meme"
        ev = e.sort_values("ignition_date").iloc[0]
        return "early" if row.entry_date < ev.ignition_date else ("reactive" if row.entry_date <= ev.peak_date else "late")
    tk["timing"] = tk.apply(tlabel, axis=1)

    print("\n  Timing vs ignition (no-meme = false positive):")
    print(tk.groupby("timing").agg(n=("ret","size"), win=("ret", lambda x:(x>0).mean()), pnl=("pnl","sum")).round(2).to_string())
    print("\n  Per-year:")
    print(tk.groupby("year").agg(n=("ret","size"), win=("ret", lambda x:(x>0).mean()), pnl=("pnl","sum")).round(2).to_string())

    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(tk["exit_date"], tk["equity"]); ax[0].axhline(BT["start_capital"], ls="--", c="gray")
    ax[0].set_title(f"Equity curve — final ${final:,.0f}"); ax[0].grid(alpha=.3)
    yr = tk.groupby("year")["pnl"].sum()
    ax[1].bar(yr.index.astype(str), yr.values, color=["#2e9e5b" if v > 0 else "#c0392b" for v in yr.values])
    ax[1].axhline(0, c="k", lw=.6); ax[1].set_title("P&L theo năm"); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

    rep = tk.assign(entry=lambda x: x.entry_date.dt.date, exit=lambda x: x.exit_date.dt.date)[
        ["ticker","entry","entry_px","exit","exit_px","days","rvol_in","dist_in","vsma_in","ret","pnl","timing"]]
    rep.to_csv(CACHE_DIR / "entry_v2_trades.csv", index=False)
    print("\n  Per-trade (15 gần nhất):"); print(rep.tail(15).to_string(index=False))
    print("\n  Đã lưu: meme_cache/entry_v2_trades.csv")


In [ ]:
# Cell 4 — TIMING INTRADAY: sau khi xác định NGÀY entry (daily) -> nhìn giá PHÚT để canh vào
# Alpaca minute (IEX free) chỉ có ~2 năm gần -> demo trên các entry GẦN NHẤT.
def fetch_minute(sym, day):
    start = pd.to_datetime(day).strftime("%Y-%m-%d")
    end = (pd.to_datetime(day) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    r = requests.get(f"https://data.alpaca.markets/v2/stocks/{sym}/bars",
                     params={"timeframe": "1Min", "start": start, "end": end, "limit": 1000, "feed": "iex"},
                     headers=ALPACA_HEADERS, timeout=15)
    if r.status_code != 200: return None
    b = r.json().get("bars") or []
    if not b: return None
    d = pd.DataFrame(b).rename(columns={"t": "ts", "o": "open", "h": "high", "l": "low", "c": "close", "v": "vol"})
    d["ts"] = pd.to_datetime(d["ts"]); return d

def intraday_plan(sym, day):
    m = fetch_minute(sym, day)
    if m is None or len(m) == 0:
        print(f"  {sym} {day}: không có minute data (quá cũ — free chỉ ~2 năm)."); return
    m["vwap"] = (m["close"] * m["vol"]).cumsum() / m["vol"].cumsum()
    cross = m[m["close"] > m["vwap"]]
    fill = cross.iloc[0] if len(cross) else m.iloc[0]    # gợi ý: vào lần đầu giá reclaim VWAP
    print(f"  {sym} {day}: open {m['close'].iloc[0]:.3f} | VWAP-reclaim @ {fill['ts'].time()} px {fill['close']:.3f} | "
          f"low {m['low'].min():.3f} high {m['high'].max():.3f}")
    try:
        plt.figure(figsize=(11, 3.4)); plt.plot(m["ts"], m["close"], lw=1, label="price")
        plt.plot(m["ts"], m["vwap"], lw=1, ls="--", label="VWAP")
        plt.scatter([fill["ts"]], [fill["close"]], c="g", zorder=5, label="gợi ý fill")
        plt.title(f"{sym} {day} — minute (canh entry)"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()
    except Exception as e:
        print("(skip plot:", e, ")")

if len(ab):
    for _, tr in ab.sort_values("entry_date").tail(3).iterrows():
        intraday_plan(tr["ticker"], pd.to_datetime(tr["entry_date"]).strftime("%Y-%m-%d"))
else:
    print("Chưa có absorption trade để demo intraday.")

## Phân tích: recall (bắt hết spike?) + tách big-win vs stopped

In [ ]:
# Cell 6 — COVERAGE/RECALL: bắt được bao nhiêu % đợt ignite? thiếu điều kiện nào?
def _ind(g):
    close = g["close"].values.astype(float); vol = g["vol"].values.astype(float); low = g["low"].values.astype(float); n = len(g)
    rvol = np.full(n, np.nan); ret1 = np.full(n, np.nan); dist = np.full(n, np.nan); vsma = np.full(n, np.nan)
    for k in range(60, n):
        a = vol[k-20:k].mean(); rvol[k] = vol[k]/a if a > 0 else np.nan
        ret1[k] = close[k]/close[k-1] - 1; dist[k] = close[k]/low[k-60:k].min() - 1; vsma[k] = close[k]/close[k-20:k].mean() - 1
    return close, rvol, ret1, dist, vsma, n

_idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt.groupby("ticker")}
P = ENTRY; WIN = 15
captured = total = condn = 0
cond_pass = {"absorb": 0, "dip": 0, "early": 0, "curl": 0, "si": 0}
for _, ev in events.iterrows():
    g = _idx.get(ev.ticker)
    if g is None: continue
    pos = g.index[g["date"] == ev.ignition_date]
    if len(pos) == 0 or pos[0] < 70: continue
    t = pos[0]; total += 1
    close, rvol, ret1, dist, vsma, n = _ind(g)
    got = False
    for i in range(max(64, t - WIN), min(t + 1, n - 1)):
        w0 = max(60, i - P["lookback"])
        absorb = any(rvol[k] > P["spike_rvol"] and abs(ret1[k]) < P["flat_max"] for k in range(w0, i))
        dip = (dist[i] < dist[i-P["dip_ago"]]) and (vsma[i] < vsma[i-P["dip_ago"]])
        early = rvol[i] < P["rvol_now_max"]; curl = close[i] > close[i-1]
        si = _si_gate_ok(ev.ticker, g["date"].iloc[i], EXIT_F)
        cond_pass["absorb"] += absorb; cond_pass["dip"] += dip; cond_pass["early"] += early
        cond_pass["curl"] += curl; cond_pass["si"] += si; condn += 1
        if absorb and dip and early and curl and si: got = True
    captured += int(got)

print(f"RECALL: bắt được {captured}/{total} đợt meme = {captured/max(total,1):.0%}  (MISS {total-captured})")
print("\nTỉ lệ THỎA mỗi điều kiện trong cửa sổ 15 ngày trước ignition (THẤP nhất = nút thắt lọc mất nhiều nhất):")
for k, v in sorted(cond_pass.items(), key=lambda x: x[1]):
    print(f"  {k:8s}: {v/max(condn,1):.0%}")
print("\n -> nếu recall thấp + 'absorb' hoặc 'si' % thấp = chính nó loại các đợt thật. Nới điều kiện đó (Cell 3) để bắt thêm.")
print("    (recall = bắt được; nhưng nới quá -> nhiều no-meme. Cân bằng recall vs precision.)")


In [ ]:
# Cell 7 — PHÂN BIỆT big-win (>50%) vs stopped (<=0): indicator nào tách 2 trường hợp?
def enrich_ab(ab_df):
    idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt.groupby("ticker")}
    out = []
    for _, tr in ab_df.iterrows():
        g = idx.get(tr["ticker"])
        if g is None: continue
        pos = g.index[g["date"] == tr["entry_date"]]
        if len(pos) == 0 or pos[0] < 70: continue
        i = pos[0]
        close = g["close"].values.astype(float); vol = g["vol"].values.astype(float)
        low = g["low"].values.astype(float); high = g["high"].values.astype(float)
        atr = _atr(high, low, close, 14)
        w0 = max(60, i - ENTRY["lookback"])
        spikes = [k for k in range(w0, i)
                  if vol[k-20:k].mean() > 0 and vol[k]/vol[k-20:k].mean() > ENTRY["spike_rvol"]
                  and abs(close[k]/close[k-1]-1) < ENTRY["flat_max"]]
        max_rvol = max((vol[k]/vol[k-20:k].mean() for k in spikes), default=np.nan)   # cú thrust mạnh cỡ nào
        flatness = min((abs(close[k]/close[k-1]-1) for k in spikes), default=np.nan)  # giá càng ĐỨNG (nhỏ) càng absorption thật
        si = si_asof(tr["ticker"], g["date"].iloc[i]); dtc = si[0] if si else np.nan
        out.append({
            "ret": tr["ret"],
            "bucket": "big_win" if tr["ret"] >= 0.5 else ("loss" if tr["ret"] <= 0 else "mid"),
            "max_rvol_spike": max_rvol, "flatness": flatness,
            "dist_in": tr["dist_in"], "vsma_in": tr["vsma_in"], "rvol_in": tr["rvol_in"],
            "atr_pct": atr[i]/close[i] if not np.isnan(atr[i]) and close[i] > 0 else np.nan,
            "price": close[i], "si_dtc": dtc,
            "days_since_spike": (i - max(spikes)) if spikes else np.nan,
        })
    return pd.DataFrame(out)

if len(ab) == 0:
    print("Chưa có trade.")
else:
    fx = enrich_ab(ab)
    feats = ["max_rvol_spike", "flatness", "dist_in", "vsma_in", "rvol_in", "atr_pct", "price", "si_dtc", "days_since_spike"]
    bw, ls = fx[fx.bucket == "big_win"], fx[fx.bucket == "loss"]
    print(f"big_win (>50%): {len(bw)} | loss (<=0): {len(ls)} | mid: {(fx.bucket=='mid').sum()}  | tổng {len(fx)}")
    if len(bw) >= 3 and len(ls) >= 3:
        print("\nTrung bình feature — big_win vs loss (khác nhau RÕ = tách được):")
        cmp = pd.DataFrame({"big_win": bw[feats].mean(), "loss": ls[feats].mean()})
        cmp["sep"] = (cmp["big_win"] - cmp["loss"]) / (cmp["loss"].abs() + 1e-9)
        print(cmp.round(3).to_string())
        print("\nTương quan feature với return:")
        print(fx[feats + ["ret"]].corr()["ret"].drop("ret").sort_values(key=lambda x: x.abs(), ascending=False).round(3).to_string())
        try:
            from sklearn.tree import DecisionTreeClassifier
            d = fx.dropna(subset=feats); d = d[d.bucket.isin(["big_win", "loss"])]
            if len(d) > 25:
                clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8).fit(d[feats], (d.bucket == "big_win").astype(int))
                imp = pd.Series(clf.feature_importances_, index=feats).sort_values(ascending=False)
                print("\nDecisionTree importance (tách big_win vs loss):"); print(imp.round(3).to_string())
                print("\n -> feature đứng đầu + sep lớn = LỌC BỔ SUNG để né loss. Thêm nó vào ENTRY (Cell 3).")
        except ImportError:
            print("\n(cài scikit-learn để có tree: %pip install scikit-learn)")
    else:
        print("Mẫu quá nhỏ để tách (cần >=3 mỗi nhóm). Nới ENTRY ở Cell 3 cho nhiều lệnh hơn.")


## Pre-spike separability + menu indicator tăng recall

In [ ]:
# Cell 8 — Tách big-win vs loss CHỈ dùng data TRƯỚC ngày spike (setup quality biết trước được không?)
def prespike_features(ab_df):
    idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt.groupby("ticker")}
    out = []
    for _, tr in ab_df.iterrows():
        g = idx.get(tr["ticker"])
        if g is None: continue
        pos = g.index[g["date"] == tr["entry_date"]]
        if len(pos) == 0 or pos[0] < 70: continue
        i = pos[0]
        close = g["close"].values.astype(float); vol = g["vol"].values.astype(float)
        low = g["low"].values.astype(float); high = g["high"].values.astype(float)
        atr = _atr(high, low, close, 14)
        # tìm NGÀY SPIKE k* (cú absorption gần nhất trước entry)
        w0 = max(60, i - ENTRY["lookback"])
        spikes = [k for k in range(w0, i)
                  if vol[k-20:k].mean() > 0 and vol[k]/vol[k-20:k].mean() > ENTRY["spike_rvol"]
                  and abs(close[k]/close[k-1]-1) < ENTRY["flat_max"]]
        if not spikes: continue
        ks = max(spikes)               # ngày spike
        if ks < 60: continue
        # === FEATURE CHỈ DÙNG DỮ LIỆU <= ks (trước/tại spike, KHÔNG nhìn sau) ===
        si = si_asof(tr["ticker"], g["date"].iloc[ks]); dtc = si[0] if si else np.nan
        out.append({
            "ret": tr["ret"],
            "bucket": "big_win" if tr["ret"] >= 0.5 else ("loss" if tr["ret"] <= 0 else "mid"),
            "pre_trend_20":   close[ks]/close[ks-20]-1,                       # xu hướng giá vào spike
            "pre_atr_pct":    atr[ks-1]/close[ks-1] if close[ks-1] > 0 else np.nan,  # vola TRƯỚC spike
            "dist_low_spike": close[ks]/low[ks-60:ks].min()-1,               # cách đáy 60d lúc spike
            "vol_base_rvol":  np.nanmean([vol[k]/vol[k-20:k].mean() for k in range(ks-10, ks)]),  # nền volume trước
            "si_dtc_pre":     dtc,                                            # nhiên liệu short (point-in-time)
            "n_prior_spikes": sum(1 for k in range(ks-40, ks) if k >= 80 and vol[k-20:k].mean() > 0 and vol[k]/vol[k-20:k].mean() > 3),
            "price_spike":    close[ks],
        })
    return pd.DataFrame(out)

if len(ab) == 0:
    print("Chưa có trade.")
else:
    fp = prespike_features(ab)
    feats = ["pre_trend_20", "pre_atr_pct", "dist_low_spike", "vol_base_rvol", "si_dtc_pre", "n_prior_spikes", "price_spike"]
    bw, ls = fp[fp.bucket == "big_win"], fp[fp.bucket == "loss"]
    print(f"Dùng CHỈ data <= ngày spike. big_win {len(bw)} | loss {len(ls)} | mid {(fp.bucket=='mid').sum()}")
    if len(bw) >= 3 and len(ls) >= 3:
        cmp = pd.DataFrame({"big_win": bw[feats].mean(), "loss": ls[feats].mean()})
        cmp["sep"] = (cmp["big_win"] - cmp["loss"]) / (cmp["loss"].abs() + 1e-9)
        print("\nTrung bình (pre-spike) big_win vs loss:"); print(cmp.round(3).to_string())
        print("\nTương quan pre-spike feature vs return:")
        print(fp[feats + ["ret"]].corr()["ret"].drop("ret").sort_values(key=lambda x: x.abs(), ascending=False).round(3).to_string())
        try:
            from sklearn.tree import DecisionTreeClassifier
            from sklearn.model_selection import cross_val_score
            d = fp.dropna(subset=feats); d = d[d.bucket.isin(["big_win", "loss"])]
            if len(d) > 30:
                y = (d.bucket == "big_win").astype(int)
                clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10)
                cv = cross_val_score(clf, d[feats], y, cv=4).mean()
                clf.fit(d[feats], y)
                print(f"\nDecisionTree (pre-spike) — CV accuracy {cv:.0%} (baseline đoán bừa = {max(y.mean(),1-y.mean()):.0%})")
                print(pd.Series(clf.feature_importances_, index=feats).sort_values(ascending=False).round(3).to_string())
                print("\n -> CV > baseline rõ = TÁCH ĐƯỢC bằng data trước spike (setup quality biết trước).")
                print("    CV ~ baseline = KHÔNG tách được trước -> kết quả phụ thuộc cái xảy ra SAU entry (squeeze có thật hay không).")
        except ImportError:
            print("\n(cài scikit-learn để có CV accuracy)")
    else:
        print("Mẫu nhỏ — nới ENTRY ở Cell 3.")


In [ ]:
# Cell 9 — TĂNG RECALL: menu indicator khác (mỗi cái recall + precision) -> chọn cái đáng thêm
def signal_arrays(g):
    close = g["close"].values.astype(float); high = g["high"].values.astype(float)
    low = g["low"].values.astype(float); vol = g["vol"].values.astype(float); n = len(g)
    atr = _atr(high, low, close, 14)
    rvol = np.full(n, np.nan); ret1 = np.full(n, np.nan)
    for k in range(20, n):
        a = vol[k-20:k].mean(); rvol[k] = vol[k]/a if a > 0 else np.nan
        ret1[k] = close[k]/close[k-1]-1
    dvol = close * vol
    obv = np.zeros(n)
    for k in range(1, n): obv[k] = obv[k-1] + (np.sign(close[k]-close[k-1]) * vol[k])
    sig = {k: np.zeros(n, dtype=bool) for k in ["absorb","dvol_surge","squeeze_break","obv_div","gap_vol"]}
    for i in range(64, n):
        # 1) absorption (đang dùng): volume spike + giá đứng trong 7 ngày
        sig["absorb"][i] = any(rvol[k] > 4 and abs(ret1[k]) < 0.04 for k in range(max(60,i-7), i))
        # 2) dollar-volume surge: $vol >> nền (tiền lớn vào, bất kể giá)
        base = dvol[i-20:i]; sig["dvol_surge"][i] = base.std() > 0 and (dvol[i]-base.mean())/base.std() > 2.5
        # 3) volatility squeeze -> expansion: ATR co lại rồi range hôm nay bung
        prior_atr = np.nanmean(atr[i-20:i-1])
        sig["squeeze_break"][i] = (not np.isnan(prior_atr)) and atr[i-1] < 0.8*prior_atr and (high[i]-low[i]) > 1.5*atr[i-1]
        # 4) OBV divergence: dòng tiền (OBV) lập đỉnh 20d nhưng GIÁ chưa -> gom âm thầm
        sig["obv_div"][i] = obv[i] >= obv[i-20:i+1].max() and close[i] < close[i-20:i].max()
        # 5) gap-up trên volume: mở cửa nhảy + volume cao
        sig["gap_vol"][i] = (close[i-1] > 0) and (low[i]/close[i-1]-1 > 0.03) and rvol[i] > 2
    return sig

idx = {s: g.sort_values("date").reset_index(drop=True) for s, g in bt.groupby("ticker")}
allsig = {s: signal_arrays(g) for s, g in idx.items()}
names = ["absorb","dvol_surge","squeeze_break","obv_div","gap_vol"]

# RECALL: % đợt meme có signal trong [ign-15, ign]
recall = {nm: 0 for nm in names}; recall_any = 0; tot = 0
for _, ev in events.iterrows():
    g = idx.get(ev.ticker); 
    if g is None: continue
    pos = g.index[g["date"] == ev.ignition_date]
    if len(pos) == 0 or pos[0] < 80: continue
    t = pos[0]; tot += 1; sg = allsig[ev.ticker]; lo = max(64, t-15)
    hit_any = False
    for nm in names:
        if sg[nm][lo:t+1].any(): recall[nm] += 1; hit_any = True
    recall_any += int(hit_any)

# PRECISION xấp xỉ: trong các ngày signal bật, % nằm trong 15 ngày trước 1 ignition
evset = {(r.ticker, pd.to_datetime(r.ignition_date)) for r in events.itertuples()}
def near_ign(sym, date):
    return any(s == sym and 0 <= (ig - date).days <= 15 for s, ig in evset)
prec = {nm: [0,0] for nm in names}
for sym, sg in allsig.items():
    g = idx[sym]
    for nm in names:
        days = np.where(sg[nm])[0]
        for i in days:
            prec[nm][1] += 1
            if near_ign(sym, g["date"].iloc[i]): prec[nm][0] += 1

print(f"Tổng {tot} đợt meme. RECALL (bắt được trong 15 ngày trước ignition) + PRECISION mỗi signal:\n")
print(f"  {'signal':14s} {'recall':>8s} {'precision':>10s}")
for nm in names:
    pr = prec[nm][0]/prec[nm][1] if prec[nm][1] else 0
    print(f"  {nm:14s} {recall[nm]/max(tot,1):>7.0%} {pr:>10.1%}")
print(f"\n  {'OR bất kỳ':14s} {recall_any/max(tot,1):>7.0%}  <- recall nếu vào khi BẤT KỲ signal nào bật")
print("\n -> recall cao = bắt nhiều đợt; precision cao = ít báo giả. Thêm signal recall cao + precision khá vào ENTRY.")
print("    Lưu ý: gom OR nhiều signal -> recall tăng nhưng no-meme cũng tăng. Chọn 1-2 cái precision tốt nhất.")
